### Cell 1: Defines `DATA` path

In [ ]:
DATA = "../data/data999.xlsx"

### Cell 2: Environment & Dependency Initialization


In [ ]:
%load_ext autoreload
%autoreload 2

import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import scikit_posthocs as sp
from scistatvisn import select_statistical_test, format_posthoc_results

sns.set_theme(style="whitegrid")
plt.rcParams["font.size"] = 11

**Technical Description:**
Configures the execution environment, path variables, dynamic module reloads, core scientific libraries, and global plotting parameters.

**Configuration:** Enables IPython `autoreload 2` to automatically refresh imported custom modules (`scistatvisn.analysis`) upon source modifications without restarting the kernel.

**Core Imports:**

- Data manipulation: `pandas`, `numpy`

- Inferential statistics: `scipy.stats`

- Post-hoc pairwise testing: `scikit_posthocs`

- Custom statistical routing pipeline: `scistatvisn.analysis.select_statistical_test`, `scistatvisn.analysis.format_posthoc_results`

**Plotting Setup:** Applies Seaborn's `whitegrid` theme and enforces a global default font size of 11pt across `Matplotlib` axes.

### Cell 3: Data Ingestion, Schema Normalization, Type Casting & Subject Aggregation

In [ ]:
import pandas as pd
import numpy as np
import warnings

COLUMN_MAPPING = {
    "subject_id": ["subject_id", "subject", "id", "animal", "animal_id", "sample", "sample_id", "patient", "patient_id"],
    "condition": ["condition", "group", "treatment/control", "experiment/control"],
    "metric_value": ["metric_value", "metric", "value", "amount", "reading"],
}

def normalize_and_validate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalizes column names based on the mapping.
    If columns are not found, positional fallback is used (0->subject_id, 1->condition, 2+->metric_value).
    """
    df = df.copy()

    renamed_cols = {}
    for col in df.columns:
        col_clean = str(col).strip().lower()
        for standard_name, aliases in COLUMN_MAPPING.items():
            if col_clean in aliases:
                renamed_cols[col] = standard_name
                break

    df = df.rename(columns=renamed_cols)

    required_cols = ["subject_id", "condition", "metric_value"]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        if len(df.columns) < 3:
            raise ValueError(
                f"Error: The table has less than 3 columns ({len(df.columns)} шт.). "
                f"Unable to automatically detect {missing_cols}."
            )

        fallback_cols = list(df.columns)

        pos_mapping = {}
        if "subject_id" in missing_cols:
            pos_mapping[fallback_cols[0]] = "subject_id"
        if "condition" in missing_cols:
            pos_mapping[fallback_cols[1]] = "condition"
        if "metric_value" in missing_cols:
            pos_mapping[fallback_cols[2]] = "metric_value"

        warnings.warn(
            f"\n[WARNING] Unable to find columns {missing_cols} by name.\n"
            f"Automatic position matching used:\n"
            f" - {fallback_cols[0]} -> subject_id\n"
            f" - {fallback_cols[1]} -> condition\n"
            f" - {fallback_cols[2]} -> metric_value\n"
            f"Make sure the column order in the file is correct!",
            UserWarning
        )

        df = df.rename(columns=pos_mapping)

    return df

df = pd.read_excel(DATA)
df = normalize_and_validate_columns(df)

if df["metric_value"].dtype == "object":
    df["metric_value"] = (
        df["metric_value"]
        .astype(str)
        .str.replace(",", ".")
        .astype(float)
    )

df_subject = df.groupby(["subject_id", "condition"], as_index=False)["metric_value"].mean()

conditions = df_subject["condition"].unique()
print(f"Detected conditions ({len(conditions)}): {list(conditions)}")
print(f"Total unique subjects: {df_subject['subject_id'].nunique()}")

df_subject.head()

**Technical Description:**
Executes data ingestion, flexible schema normalization, type casting, and primary aggregation at the individual subject level.

* **Data Parsing & Column Mapping:** Imports the raw Excel dataset. Normalizes column headers via flexible alias matching (`COLUMN_MAPPING`) supporting multi-language variants (`subject_id`, `condition`, `metric_value`). Fallbacks to positional column mapping $(0 \to \text{subject\_id}, 1 \to \text{condition}, 2 \to \text{metric\_value})$ with a explicit runtime warning if column names are unrecognized.
* **Cleaning & Type Casting:** Identifies localized numerical formatting, replacing decimal commas with dots (`,` $\to$ `.`), and casts `metric_value` to `float64`.
* **Subject-Level Aggregation:** Computes the mean metric value per subject and condition:
  $$\bar{x}_{i, k} = \frac{1}{n_{i, k}} \sum_{m=1}^{n_{i, k}} x_{i, k, m}$$
  where $x_{i, k, m}$ is the $m$-th measurement of subject $i$ under condition $k$.
* **Diagnostics:** Reports detected experimental conditions, total sample size ($N$), and previews `df_subject`.

### Cell 4: Test Criteria Selection, Hypothesis Testing & Post-Hoc Analysis

In [ ]:
test_info = select_statistical_test(df_subject, num_groups_mode="auto")

rec_test = test_info["recommended_test"]
rec_post_hoc = test_info["recommended_post_hoc"]
num_groups = test_info["actual_num_groups"]

print(f"Applied Mode: {test_info['mode_used']} ({num_groups} groups)")
print(f"Recommended Test: {rec_test}")
if rec_post_hoc:
    print(f"Recommended Post-Hoc: {rec_post_hoc}")

if num_groups == 2:
    # comparison of 2 groups:
    df_pivoted = df_subject.pivot(
        index="subject_id", columns="condition", values="metric_value"
    ).dropna()
    g1_col, g2_col = conditions[0], conditions[1]
    g1_vals = df_pivoted[g1_col]
    g2_vals = df_pivoted[g2_col]
    n1, n2 = len(g1_vals), len(g2_vals)

    if rec_test == "wilcoxon":
        res = stats.wilcoxon(g1_vals, g2_vals)
        stat, p_val = res.statistic, res.pvalue
        stat_col_name, test_title_name = "W-statistic", "Wilcoxon Signed-Rank Test"
        z_score = abs(stats.norm.ppf(p_val / 2)) if p_val > 0 else 0
        effect_size = z_score / np.sqrt(n1)
        effect_metric = "Effect Size (r)"
        labels = {0.5: "Large effect", 0.3: "Medium effect", 0.1: "Small effect"}

    elif rec_test == "mannwhitneyu":
        stat, p_val = stats.mannwhitneyu(g1_vals, g2_vals, alternative="two-sided")
        stat_col_name, test_title_name = "U-statistic", "Mann-Whitney U Test"
        effect_size = abs(1 - (2 * stat) / (n1 * n2))
        effect_metric = "Effect Size (r)"
        labels = {0.5: "Large effect", 0.3: "Medium effect", 0.1: "Small effect"}

    elif rec_test == "ttest_rel":
        stat, p_val = stats.ttest_rel(g1_vals, g2_vals)
        stat_col_name, test_title_name = "t-statistic", "Paired Student's t-test"
        diff = g1_vals - g2_vals
        std_diff = np.std(diff, ddof=1)
        effect_size = np.abs(np.mean(diff) / std_diff) if std_diff > 0 else 0.0
        effect_metric = "Cohen's d"
        labels = {0.8: "Large effect", 0.5: "Medium effect", 0.2: "Small effect"}

    else:  # ttest_ind
        stat, p_val = stats.ttest_ind(g1_vals, g2_vals)
        stat_col_name, test_title_name = "t-statistic", "Independent Student's t-test"
        s_pooled = np.sqrt(
            ((n1 - 1) * np.var(g1_vals, ddof=1) + (n2 - 1) * np.var(g2_vals, ddof=1))
            / (n1 + n2 - 2)
        )
        effect_size = (
            np.abs(np.mean(g1_vals) - np.mean(g2_vals)) / s_pooled if s_pooled > 0 else 0.0
        )
        effect_metric = "Cohen's d"
        labels = {0.8: "Large effect", 0.5: "Medium effect", 0.2: "Small effect"}

    cutoffs = sorted(labels.keys(), reverse=True)
    if effect_size >= cutoffs[0]:
        effect_label = labels[cutoffs[0]]
    elif effect_size >= cutoffs[1]:
        effect_label = labels[cutoffs[1]]
    elif effect_size >= cutoffs[2]:
        effect_label = labels[cutoffs[2]]
    else:
        effect_label = "Negligible effect"

    effect_str = f"{effect_size:.2f} ({effect_label})"

    if p_val < 0.001:
        sig_text = "p < 0.001 ***"
    elif p_val < 0.01:
        sig_text = "p < 0.01 **"
    elif p_val < 0.05:
        sig_text = "p < 0.05 *"
    else:
        sig_text = f"p = {p_val:.3f} (ns)"

    # Summary table for 2 groups
    results_df = pd.DataFrame(
        {
            "Group": [g1_col, g2_col],
            "N (Subjects)": [n1, n2],
            "Median [IQR]": [
                f"{g1_vals.median():.2f} [{g1_vals.quantile(0.75) - g1_vals.quantile(0.25):.2f}]",
                f"{g2_vals.median():.2f} [{g2_vals.quantile(0.75) - g2_vals.quantile(0.25):.2f}]",
            ],
            "Mean ± SD": [
                f"{g1_vals.mean():.2f} ± {g1_vals.std():.2f}",
                f"{g2_vals.mean():.2f} ± {g2_vals.std():.2f}",
            ],
            stat_col_name: [f"{stat:.2f}", "—"],
            "p-value": [f"{p_val:.6f}", "—"],
            "Significance": [sig_text, "—"],
            effect_metric: [effect_str, "—"],
        }
    )
    display(results_df)

else:
    # Comparison of 3+ groups:
    grouped_data = [g["metric_value"].values for _, g in df_subject.groupby("condition")]

    if rec_test == "kruskal_wallis":
        stat, p_val = stats.kruskal(*grouped_data)
        stat_col_name, test_title_name = "H-statistic", "Kruskal-Wallis Test"
    elif rec_test == "anova_oneway":
        stat, p_val = stats.f_oneway(*grouped_data)
        stat_col_name, test_title_name = "F-statistic", "One-Way ANOVA"
    elif rec_test == "friedman":
        df_piv = df_subject.pivot(
            index="subject_id", columns="condition", values="metric_value"
        ).dropna()
        stat, p_val = stats.friedmanchisquare(*[df_piv[col] for col in df_piv.columns])
        stat_col_name, test_title_name = "Q-statistic", "Friedman Test"
    else:  # rm_anova
        stat, p_val = 0.0, 0.0
        stat_col_name, test_title_name = "F-statistic", "Repeated Measures ANOVA"

    print(f"Global Test: {test_title_name} | {stat_col_name}: {stat:.2f} | p-value: {p_val:.6f}")

    # Post-hoc analysis in the case of significant difference (p < 0.05)
    if p_val < 0.05:
        print("\nPerforming Post-Hoc Analysis...")
        if rec_post_hoc in ["dunn_test", "nemenyi_or_dunn"]:
            posthoc_matrix = sp.posthoc_dunn(
                df_subject, val_col="metric_value", group_col="condition", p_adjust="holm"
            )
        else:
            posthoc_matrix = sp.posthoc_tukey(
                df_subject, val_col="metric_value", group_col="condition"
            )

        posthoc_df = format_posthoc_results(posthoc_matrix)
        display(posthoc_df)
    else:
        print(
            "\nNo statistically significant differences found between groups (p >= 0.05). Post-hoc omitted."
        )

**Technical Description:**
Dynamically routes data through an automated decision workflow via `scistatvisn.analysis.select_statistical_test`, computes global and pairwise statistical tests, calculates standardized effect sizes, and outputs structured summary matrices.

---

#### 1. Dynamic Criterion Selection (`select_statistical_test`)
The decision pipeline in `analysis.py` evaluates the structural properties and distributional assumptions of `df_subject` to automatically select the appropriate test based on:
* **Group Cardinality ($k$):** Differentiates between two-group designs ($k = 2$) and multi-group designs ($k \ge 3$).
* **Experimental Design:** Detects paired (repeated measures) versus independent samples based on `subject_id` alignment across conditions.
* **Parametric Assumptions:** Evaluates sample normality (e.g., Shapiro-Wilk test) and variance homogeneity (e.g., Levene's test) to choose between parametric and non-parametric estimators.

---

#### 2. Statistical Calculations for $k = 2$ Groups

##### A. Non-parametric Paired: Wilcoxon Signed-Rank Test (`wilcoxon`)
* **Test Statistic ($W$):** Sum of signed ranks based on paired differences $d_i = x_{i,1} - x_{i,2}$.
* **Effect Size ($r$):** Derived from the asymptotic standard normal quantile $z$:
  $$r = \frac{\vert{}z\vert{}}{\sqrt{N}}$$
  *Effect Thresholds:* $r \ge 0.1$ (Small), $r \ge 0.3$ (Medium), $r \ge 0.5$ (Large).

##### B. Non-parametric Independent: Mann-Whitney U Test (`mannwhitneyu`)
* **Test Statistic ($U$):** Rank-sum statistic evaluating distribution shifts between sample sizes $n_1$ and $n_2$.
* **Effect Size (Rank-Biserial Correlation $r$):**
  $$r = \left\vert{} 1 - \frac{2U}{n_1 \cdot n_2} \right\vert{}$$
  *Effect Thresholds:* $r \ge 0.1$ (Small), $r \ge 0.3$ (Medium), $r \ge 0.5$ (Large).

##### C. Parametric Paired: Paired Student's t-test (`ttest_rel`)
* **Test Statistic ($t$):** Evaluates mean difference $\bar{d}$ relative to the standard error of differences:
  $$t = \frac{\bar{d}}{s_d / \sqrt{N}}$$
* **Effect Size (Cohen's $d_z$):**
  $$d = \frac{\vert{}\bar{d}\vert{}}{s_d}, \quad \text{where } s_d = \sqrt{\frac{1}{N-1} \sum_{i=1}^{N} (d_i - \bar{d})^2}$$
  *Effect Thresholds:* $d \ge 0.2$ (Small), $d \ge 0.5$ (Medium), $d \ge 0.8$ (Large).

##### D. Parametric Independent: Independent Student's t-test (`ttest_ind`)
* **Test Statistic ($t$):** Compares group means $\bar{x}_1$ and $\bar{x}_2$ against pooled variance $s_{\text{pooled}}^2$.
* **Effect Size (Cohen's $d_s$):**
  $$d = \frac{\vert{}\bar{x}_1 - \bar{x}_2\vert{}}{s_{\text{pooled}}}, \quad \text{where } s_{\text{pooled}} = \sqrt{\frac{(n_1 - 1)s_1^2 + (n_2 - 1)s_2^2}{n_1 + n_2 - 2}}$$
  *Effect Thresholds:* $d \ge 0.2$ (Small), $d \ge 0.5$ (Medium), $d \ge 0.8$ (Large).

---

#### 3. Statistical Calculations for $k \ge 3$ Groups

* **Omnibus Tests:**
  * **Non-parametric Independent:** Kruskal-Wallis test ($H$-statistic via `stats.kruskal`).
  * **Parametric Independent:** One-Way ANOVA ($F$-statistic via `stats.f_oneway`).
  * **Non-parametric Paired:** Friedman test ($Q$-statistic via `stats.friedmanchisquare`).
  * **Parametric Paired:** Repeated Measures ANOVA ($F$-statistic).
* **Post-Hoc Pairwise Analysis ($p < 0.05$):**
  * **Non-parametric:** Executes Dunn's test (`sp.posthoc_dunn`) using Holm-Bonferroni correction (`p_adjust="holm"`) to control the Family-Wise Error Rate (FWER):
    $$p_{\text{adjusted}, (i)} = \min \left( 1, \max_{j \le i} \left[ (m - j + 1) p_{(j)} \right] \right)$$
  * **Parametric:** Executes Tukey's Honest Significant Difference test (`sp.posthoc_tukey`).

### Cell 5: Statistical Visualization Pipeline

In [ ]:
plt.figure(figsize=(8, 6))

ax = sns.boxplot(
    data=df_subject,
    x="condition",
    y="metric_value",
    hue="condition",
    palette="Set2",
    legend=False,
    width=0.4,
    boxprops=dict(alpha=0.8),
)

if test_info["is_paired"]:
    # For paired groups (->connecting lines):
    piv_df = df_subject.pivot(
        index="subject_id", columns="condition", values="metric_value"
    ).dropna()
    x_coords = range(len(conditions))
    for subj_id, row in piv_df.iterrows():
        plt.plot(
            x_coords,
            [row[cond] for cond in conditions],
            color="gray",
            alpha=0.35,
            linestyle="--",
            linewidth=1,
        )

    sns.stripplot(
        data=df_subject,
        x="condition",
        y="metric_value",
        color="black",
        alpha=0.8,
        size=6,
        jitter=False,
    )
else:
    # For independent groups (dots with jitter)
    sns.stripplot(
        data=df_subject,
        x="condition",
        y="metric_value",
        color="black",
        alpha=0.7,
        size=6,
        jitter=0.2,
    )

plt.title(
    f"Comparison across Conditions ({test_title_name})", fontsize=12, fontweight="bold", pad=15
)
plt.xlabel("Condition", fontsize=11, fontweight="bold")
plt.ylabel("Metric Value (Mean per Subject)", fontsize=11, fontweight="bold")

# p-values / stars
y_max = df_subject["metric_value"].max()
y_min = df_subject["metric_value"].min()
h = (y_max - y_min) * 0.05
current_y = y_max + h

if num_groups == 2:
    # For 2 groups
    plt.plot(
        [0, 0, 1, 1], [current_y, current_y + h, current_y + h, current_y], color="black", lw=1.2
    )
    plt.text(
        0.5,
        current_y + 1.2 * h,
        sig_text,
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        color="darkred",
    )
    plt.ylim(y_min - 2 * h, current_y + 4 * h)

else:
    # For 3+ groups
    if p_val < 0.05 and "posthoc_matrix" in locals():
        cond_list = list(conditions)

        for i in range(len(cond_list)):
            for j in range(i + 1, len(cond_list)):
                g1, g2 = cond_list[i], cond_list[j]
                p_pair = posthoc_matrix.loc[g1, g2]

                if p_pair < 0.05:
                    if p_pair < 0.001:
                        stars = "***"
                    elif p_pair < 0.01:
                        stars = "**"
                    else:
                        stars = "*"

                    x1, x2 = i, j
                    plt.plot(
                        [x1, x1, x2, x2],
                        [current_y, current_y + h, current_y + h, current_y],
                        color="black",
                        lw=1.2,
                    )
                    plt.text(
                        (x1 + x2) * 0.5,
                        current_y + 0.8 * h,
                        stars,
                        ha="center",
                        va="bottom",
                        fontsize=11,
                        fontweight="bold",
                        color="darkred",
                    )

                    current_y += 2.5 * h

        plt.ylim(y_min - 2 * h, current_y + 2 * h)
    else:
        plt.ylim(y_min - 2 * h, y_max + 3 * h)

plt.tight_layout()
plt.show()

**Technical Description:**
Generates publication-ready figures combining parametric/non-parametric distributional summaries, individual subject observations, paired longitudinal trajectories, and automated significance annotation brackets.

---

#### 1. Visual Layers & Distributional Metrics

* **Boxplot Base Layer:** Displays summary statistics per condition:
  * Center line: Median ($\tilde{x} = Q_2$).
  * Box limits: Interquartile Range ($\text{IQR} = Q_3 - Q_1$).
  * Whiskers: Upper and lower bounds extending to $\min(x_{\max}, Q_3 + 1.5 \cdot \text{IQR})$ and $\max(x_{\min}, Q_1 - 1.5 \cdot \text{IQR})$.
* **Individual Subject Tracking:**
  * **Paired Design (`is_paired=True`):** Connects repeated measurements for subject $i$ across conditions using dashed trajectories:
    $$\mathbf{y}_i = \left( x_{i, c_1}, x_{i, c_2}, \dots, x_{i, c_k} \right)$$
  * **Independent Design (`is_paired=False`):** Adds a jittered scatter layer (`stripplot`, $\delta_{\text{jitter}} = 0.2$) to visualize raw data density without point overlap.

---

#### 2. Dynamic Significance Annotation Algorithm

Calculates vertical bracket geometry based on data range parameters where $h = 0.05 \cdot (y_{\max} - y_{\min})$:

* **Two-Group Case ($k = 2$):** Draws a single bracket at height $y_{\text{bracket}} = y_{\max} + h$ labeled with formatted significance $p$-value text.
* **Multi-Group Case ($k \ge 3$):** Iterates over upper-triangular pairwise $p$-value matrix cells $p_{i,j}$ (for $i < j$). If $p_{i,j} < 0.05$, a bracket is drawn between $x_i$ and $x_j$ at incremented height steps:
  $$y_{\text{next}} = y_{\text{current}} + 2.5 \cdot h$$
* **Significance Mapping:**
  $$\text{Label} = \begin{cases} \text{“***”} & \text{if } p < 0.001 \\ \text{“**”} & \text{if } p < 0.01 \\ \text{“*”} & \text{if } p < 0.05 \end{cases}$$